# Research workflows
Run with the project environment. These small checks teach the workflow; study conclusions require predeclared, repeated-seed experiments.

In [ ]:
from pathlib import Path
import os, json
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
os.chdir(root)
from dynforecast.experiments import run_experiment
from dynforecast.experiments.tuning import tune
from dynforecast.visualization.dashboard import generate_dashboard
import pandas as pd
import matplotlib.pyplot as plt

## Same target, different observations
Keep position as the target while changing whether velocity is observed. Training-only preprocessing and split-local targets are applied automatically.

In [ ]:
runs = []
for inputs in [[0], [0, 1]]:
    result = run_experiment({'dataset': {'name': 'oscillator', 'n': 260}, 'model': {'name': 'lstm', 'epochs': 4, 'width': 8}, 'inputs': inputs, 'targets': [0], 'history': 16, 'horizon': 8, 'output': 'results/notebook_research'})
    assert result['status'] == 'completed', result.get('error')
    runs.append(result)
pd.DataFrame([{'inputs': r['config']['inputs'], 'rmse': r['metrics']['rmse'], 'fit_seconds': r['metrics']['fit_seconds']} for r in runs])

## Forecast window in physical units
These are measured histories and held-out future targets, rather than illustrative forecast lines.

In [ ]:
import numpy as np
r = runs[-1]
with np.load(Path(r['config']['output']) / r['run_id'] / 'predictions.npz') as a:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(a['history_times'][0], a['history'][0,:,0], color='#9aa8ba', label='History')
    ax.plot(a['target_times'][0], a['truth'][0,:,0], color='#17263d', label='Observed future')
    ax.plot(a['target_times'][0], a['predictions'][0,:,0], color='#3766df', label='LSTM forecast')
    ax.set(xlabel='Time', ylabel='Position'); ax.legend(); fig.tight_layout()

## Validation-only selection
The optimizer sees validation scores. The selected settings are then frozen for one test evaluation; SQLite preserves the study.

In [ ]:
selection = tune({'dataset': {'n': 240}, 'model': {'name': 'ridge'}, 'history': 16, 'horizon': 8, 'output': 'results/notebook_research', 'tuning': {'trials': 2, 'budget_seconds': 30, 'space': {'model.alpha': [0.01, 1.0]}}})
assert selection['status'] == 'completed'
selection['best_parameters'], selection['best_validation_rmse'], selection['test_metrics']['rmse']

## Unseen trajectories and latent forecasts
Train/validation/test trajectories remain separate. A latent forecast state need not equal the physical hidden state.

In [ ]:
latent = run_experiment({'dataset': {'name': 'oscillator', 'n': 160, 'trajectories': 5}, 'model': {'name': 'latent_ode', 'epochs': 3, 'width': 8}, 'inputs': [0], 'targets': [0], 'history': 12, 'horizon': 4, 'output': 'results/notebook_research'})
assert latent['status'] == 'completed', latent.get('error')
latent['metrics']['trajectory_rmse']

## Open the offline atlas
Use the generated HTML page in a browser. The full demonstration atlas includes all model families and matched sweeps.

In [ ]:
dashboard = generate_dashboard('results/notebook_research')
print(dashboard.resolve())